In [ ]:
import gc
import os
import zipfile
import torch
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType

# 1. Cài đặt môi trường
os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 2. Đọc và chuẩn bị dữ liệu
file_path = "/kaggle/input/all-subtask/train.xlsx" # Đảm bảo file tồn tại
df = pd.read_excel(file_path)
df = df.iloc[:, 0:2]
df.columns = ["subtask", "step"]

dataset = Dataset.from_pandas(df)
dataset = dataset.shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.111)
dataset = DatasetDict({"train": dataset["train"], "valid": dataset["test"]})

In [ ]:
file_path = "/kaggle/input/all-subtask/train.xlsx"
df = pd.read_excel(file_path)
df = df.iloc[:, 0:2]
df.columns = ["subtask", "step"]

dataset = Dataset.from_pandas(df)
dataset = dataset.shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.111)
dataset = DatasetDict({"train": dataset["train"], "valid": dataset["test"]})

model_name = "google/mt5-large"
# 2. Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. LoRA Configuration
# Với Flan-T5, q và v là quan trọng nhất, nhưng thêm k và o giúp hội tụ tốt hơn
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "k", "v", "o"] 
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.to(device)

# Câu tiếng Việt của bạn
test_text = """
    "{", "}", "[", "]", 
    "à", "á", "ạ", "ả", "ã", "â", "ầ", "ấ", "ậ", "ẩ", "ẫ", "ă", "ằ", "ắ", "ặ", "ẳ", "ẵ",
    "è", "é", "ẹ", "ẻ", "ẽ", "ê", "ề", "ế", "ệ", "ể", "ễ",
    "ì", "í", "ị", "ỉ", "ĩ",
    "ò", "ó", "ọ", "ỏ", "õ", "ô", "ồ", "ố", "ộ", "ổ", "ỗ", "ơ", "ờ", "ớ", "ợ", "ở", "ỡ",
    "ù", "ú", "ụ", "ủ", "ũ", "ư", "ừ", "ứ", "ự", "ử", "ữ",
    "ỳ", "ý", "ỵ", "ỷ", "ỹ",
    "đ", "À", "Á", "Ạ", "Ả", "Ã", "Â", "Ầ", "Ấ", "Ậ", "Ẩ", "Ẫ", "Ă", "Ằ", "Ắ", "Ặ", "Ẳ", "Ẵ",
    "È", "É", "Ẹ", "Ẻ", "Ẽ", "Ê", "Ề", "Ế", "Ệ", "Ể", "Ễ",
    "Ì", "Í", "Ị", "Ỉ", "Ĩ",
    "Ò", "Ó", "Ọ", "Ỏ", "Õ", "Ô", "Ồ", "Ố", "Ộ", "Ổ", "Ỗ", "Ơ", "Ờ", "Ớ", "Ợ", "Ở", "Ỡ",
    "Ù", "Ú", "Ụ", "Ủ", "Ũ", "Ư", "Ừ", "Ứ", "Ự", "Ử", "Ữ",
    "Ỳ", "Ý", "Ỵ", "Ỷ", "Ỹ", "Đ"
"""
tokens = tokenizer.tokenize(test_text)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = tokenizer.encode(test_text)
print(f"Tokens after adding special tokens: {tokens}")
print(f"IDs: {ids}")
print(f"Encoded IDs: {input_ids}")
print(f"Decoded: {tokenizer.decode(input_ids)}")

print("\n=== KIỂM TRA MỘT SỐ VÍ DỤ TỪ CỘT 'subtask' ===")
sample_subtasks = dataset["train"]["subtask"][:5]

for i, text in enumerate(sample_subtasks):
    print(f"\nVí dụ {i+1}: '{text}'")
    tokens = tokenizer.tokenize(text)
    input_ids = tokenizer.encode(text)
    decoded_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    unknown_tokens = [token for token in decoded_tokens if tokenizer.unk_token in token]  
    print(f"  Tokens: {tokens}")
    print(f"  Encoded IDs: {input_ids}")
    print(f"  Unknown Tokens: {unknown_tokens}")

In [ ]:
max_input_length = 128
max_target_length = 256

def preprocess_function(examples):
    inputs = ["action2json: " + str(x) for x in examples["subtask"]]
    targets = [str(x) for x in examples["step"]]
    
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True, padding=False)
    
    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True, padding=False)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Lọc các mẫu quá dài để tránh OOM (Out of Memory)
def filter_long(example):
    input_ids = tokenizer.encode("action2json: " + str(example["subtask"]), truncation=False)
    label_ids = tokenizer.encode(str(example["step"]), truncation=False)
    return len(input_ids) <= max_input_length and len(label_ids) <= max_target_length

dataset = dataset.filter(filter_long)
tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=["subtask", "step"])

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

In [ ]:
class PredictionCallback(TrainerCallback):
    def __init__(self, tokenizer, sample_texts):
        self.tokenizer = tokenizer
        self.sample_texts = sample_texts

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n{'='*20} DỰ ĐOÁN TẠI EPOCH {state.epoch:.0f} {'='*20}")
        model = kwargs['model']
        model.eval()
        
        with torch.no_grad():
            for i, text in enumerate(self.sample_texts):
                prompt = "action2json: " + text
                inputs = self.tokenizer(prompt, return_tensors="pt").to(model.device)
                generated_ids = model.generate(
                    **inputs, 
                    max_length=max_target_length, 
                    num_beams=4, 
                    early_stopping=True
                )
                decoded = self.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                print(f"[{i+1}] Input: {text}\n    Output: {decoded}")
        model.train()

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-large-action2json",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8, 
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_steps=300,
    logging_steps=300,
    eval_strategy="steps",
    eval_steps=300,
    save_strategy="steps",
    save_steps=300,
    fp16=True, 
    optim="adamw_torch",
    load_best_model_at_end=True,
    report_to="none"
)

sample_texts = [
    "Xác minh rằng liên kết 'Giới thiệu' có đổ bóng khối với độ dịch chuyển 1px.",
    "Kiểm tra xem nút 'Đăng ký' có màu nền là #ffffff không.",
    "Xác minh rằng biểu tượng loa hiển thị mức âm lượng 75%."
]

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[PredictionCallback(tokenizer, sample_texts)]
)

In [ ]:
# Giải phóng bộ nhớ trước khi chạy
torch.cuda.empty_cache()
gc.collect()

trainer.train()

In [ ]:
# Lưu Adapter
adapter_dir = "./mt5-large-lora-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# Hợp nhất và lưu Full Model
try:
    print("Merging LoRA layers...")
    merged_model = model.merge_and_unload()
    full_dir = "./mt5-large-merged"
    merged_model.save_pretrained(full_dir)
    tokenizer.save_pretrained(full_dir)
    print(f"✅ Full model saved to: {full_dir}")
except Exception as e:
    print(f"❌ Error merging: {e}")

# Hàm zip để tải về
def zip_folder(folder_path, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for file in files:
                zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), folder_path))

zip_folder(full_dir, "mt5_action2json_full.zip")
print("✅ Zipped successfully!")